## IMBb Movie Recommendation Data Preprocessing and Model Building

##### Install the Required Libraries
This command installs all the required Python libraries needed for data manipulation, text preprocessing, feature extraction, machine learning, visualization, and recommendation model development.

In [75]:
!pip install pandas numpy nltk contractions emoji scikit-learn matplotlib seaborn pickle

ERROR: Could not find a version that satisfies the requirement pickle (from versions: none)

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for pickle


##### Importing the Required Libraries

These libraries provide the necessary functions for data handling, text cleaning, tokenization, stop-word removal, lemmatization, TF-IDF feature extraction, and similarity calculation.

In [76]:
# Data Loading
import pandas as pd
import numpy as np

# Data Preprocessing
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import contractions
import emoji

# Model Building
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import pickle

##### Download Required NLTK Resources
These resources provide linguistic datasets required for text preprocessing tasks such as tokenization, stop-word removal, and lemmatization.

stopwords → Common words to remove.
punkt → Tokenizer for splitting text into words.
wordnet → Lexical database used for lemmatization.
punkt_tab → Additional tokenizer data required by recent NLTK versions.

In [77]:
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\NITHYANANTHAM.G\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\NITHYANANTHAM.G\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\NITHYANANTHAM.G\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\NITHYANANTHAM.G\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

##### Initialize the Lemmatizer

i. `WordNetLemmatizer()` is used to convert words to their base (dictionary) form, reducing different variations of a word to a common root. This helps improve text consistency and enhances the performance of text analysis and recommendation systems.

In [78]:
lemmatizer = WordNetLemmatizer()

##### Define the Text Preprocessing Function

i. The `text_processing()` function is used to clean and preprocess textual data before feature extraction.

ii. It converts all text to lowercase to maintain consistency.

iii. It expands contractions (e.g., "don't" → "do not") using the `contractions` library.

iv. It converts emojis into their text descriptions using the `emoji` library.

v. It removes special characters, punctuation marks, and numerical values using regular expressions.

vi. It retains only alphabetic words to ensure clean textual content.

vii. It tokenizes the text into individual words using `word_tokenize()`.

viii. It removes English stop words such as "the", "is", and "and" to reduce noise in the text.

ix. It applies lemmatization using `WordNetLemmatizer()` to convert words into their base forms.

x. Finally, it returns the cleaned and preprocessed text as a single string, which can be used for feature extraction and recommendation model building.

In [79]:
def text_processing(text):
    # Text Cleaning
    text = str(text).lower()
    text = contractions.fix(text)
    text = emoji.demojize(text)
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = ' '.join([word for word in text.split() if word.isalpha()])

    # Tokenization
    tokens = word_tokenize(text)

    # Removing stop words and Lemitization
    output_text = ' '.join([lemmatizer.lemmatize(
        word) for word in tokens if word not in stopwords.words('english')])

    return output_text

#####  Load dataset

In [80]:
df = pd.read_csv("imdb_movies_2024.csv")

In [81]:
df.head()

,Title,Rating,Votes,Duration,Storyline
0,1. The Ministry of Ungentlemanly Warfare,6.8,(160K),2h 2m,The British military recruits a small group of...
1,2. Dune: Part Two,8.4,(760K),2h 46m,Paul Atreides unites with the Fremen while on ...
2,3. The Substance,7.2,(439K),2h 21m,A fading celebrity takes a black-market drug: ...
3,4. Furiosa: A Mad Max Saga,7.5,(321K),2h 28m,After being snatched from the Green Place of M...
4,5. Gladiator II,6.4,(302K),2h 28m,After his home is conquered by the tyrannical ...


In [82]:
df.shape

(5088, 5)

In [83]:
df.isna().sum()

Title        0
Rating       0
Votes        0
Duration     0
Storyline    0
dtype: int64

In [84]:
df.duplicated().sum()

np.int64(0)

In [85]:
df['Title'].head()

0    1. The Ministry of Ungentlemanly Warfare
1                           2. Dune: Part Two
2                            3. The Substance
3                  4. Furiosa: A Mad Max Saga
4                             5. Gladiator II
Name: Title, dtype: object

In [86]:
df['Title'] = (df['Title']
               .str.replace(r'^\d+\.\s*', '', regex=True)
               .str.replace(r'\s*-\s*\d+$', '', regex=True))

##### Apply cleaning ONLY to storyline (better NLP)

In [87]:
df['content'] = (
    df['Title'].fillna('') + ' ' +df['Storyline'].fillna(''))

In [88]:
df['content'] = df['content'].apply(text_processing)

In [89]:
df.to_csv('cleaned_movies.csv',index=False)

##### TF-IDF Training

| Parameter           | Purpose                                      |
| ------------------- | -------------------------------------------- |
| `max_features=5000` | Limits feature count and improves efficiency |
| `ngram_range=(1,2)` | Captures single words and meaningful phrases |


In [90]:
tfidf = TfidfVectorizer(max_features=5000,ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(df['content'])

#####  Save model

In [91]:
with open("movie_recommendation_model.pkl", "wb") as f:
    pickle.dump({"tfidf": tfidf,"matrix": tfidf_matrix,"data": df}, f)

print("Model saved successfully with cleaned text!")

Model saved successfully with cleaned text!


#### Recommend Similar Movies Based on User Query

In [94]:
def recommendations_movie(query):
    query_clean = text_processing(query)
    query_vector = tfidf.transform([query_clean])
    similarities = cosine_similarity(query_vector, tfidf_matrix).flatten()
    top_indices = similarities.argsort()[::-1][:5]
    recommendations = df.iloc[top_indices].copy()
    recommendations["Similarity_Score"] = similarities[top_indices]
    recommendations.index = range(1, len(recommendations) + 1)
    return recommendations.drop('content', axis=1)

In [96]:
recommendations_movie('Amit navigates love amidst Mayas influence an').columns

Index(['Title', 'Rating', 'Votes', 'Duration', 'Storyline',
       'Similarity_Score'],
      dtype='object')